In [ ]:
import pandas as pd

roll_number = "1024170415"

fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
     "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
     "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
     "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.",
     "keywords": "pay payment upi fee", "category": "billing"},
]

pool = {
    "billing": [
        {"question": "how do i get a refund", "answer": "Refunds are processed within 7 working days to the original payment method.",
         "keywords": "refund money return"},
        {"question": "where can i download my invoice", "answer": "Go to Billing > Invoices and click Download.",
         "keywords": "invoice receipt download"},
    ],
    "account": [
        {"question": "how do i update my registered mobile number", "answer": "Go to Profile > Contact Details and verify the new number with an OTP.",
         "keywords": "mobile number update"},
        {"question": "how do i delete my account", "answer": "Go to Settings > Privacy > Delete Account.",
         "keywords": "delete close account"},
    ],
    "general": [
        {"question": "where is your office located", "answer": "Our office is at the main campus, Block A.",
         "keywords": "office location address"},
        {"question": "how can i contact support", "answer": "Email support@example.com or call the helpline.",
         "keywords": "contact support helpline"},
    ],
}

categories = ["billing", "account", "general"]
last_two = [int(d) for d in roll_number if d.isdigit()][-2:]

custom_entries = []
used = {c: 0 for c in categories}
for d in last_two:
    cat = categories[d % 3]
    entry = dict(pool[cat][used[cat] % len(pool[cat])])
    entry["category"] = cat
    used[cat] += 1
    custom_entries.append(entry)

df = pd.DataFrame(fixed_entries + custom_entries)
print(df.to_string())

In [ ]:
stopwords = {"what", "is", "the", "how", "to", "can", "i", "do", "are", "your", "my", "a", "of", "where", "you"}

def tokens(text):
    return {w for w in text.lower().split() if w not in stopwords}

def score_entries(query, df):
    q = tokens(query)
    rows = []
    for i, row in df.iterrows():
        entry_words = tokens(row["keywords"]) | tokens(row["question"])
        score = len(q & entry_words) / len(q) if q else 0
        rows.append((i, score))
    result = df.copy()
    result["confidence"] = [s for _, s in rows]
    result = result[result["confidence"] > 0].sort_values("confidence", ascending=False)
    return result

print(score_entries("how can i pay the annual fee", df)[["question", "answer", "confidence"]].to_string())

In [ ]:
def same_category(category_name, df):
    return df[df["category"] == category_name]["question"].tolist()

target_category = df.iloc[-1]["category"]
print(target_category)
for q in same_category(target_category, df):
    print("-", q)

In [ ]:
new_keyword = input("Enter a new keyword: ").strip()
df.loc[0, "keywords"] = df.loc[0, "keywords"] + " " + new_keyword
df.to_csv(f"{roll_number}_faq_data.csv", index=False)
print(df.to_string())

In [ ]:
print(df.groupby("category").size())

In [ ]:
def best_matches(query, df):
    ranked = score_entries(query, df)
    if ranked.empty:
        print("No match found")
        return ranked
    top = ranked["confidence"].max()
    best = ranked[ranked["confidence"] == top]
    if len(best) == 1:
        print("Best match:")
    else:
        print("Tie between", len(best), "equally good matches:")
    for _, row in best.iterrows():
        print(f"Q: {row['question']}\nA: {row['answer']}\nConfidence: {row['confidence']:.2f}\n")
    return best

best_matches("fee", df)
best_matches("how to reset password", df)